In [12]:
import pickle
import numpy as np
import pandas as pd

In [22]:
# 1. Load user's FP&A file (CSV or Excel)
input_path=input("Enter file path(.csv or xlsx:)").strip()

if input_path.endswith('.csv'):
    data=pd.read_csv(input_path)
    print("File sucessfully downloaded with CSV format")
else:
    data=pd.read_csv(input_path)
    print("File sucessfully downloaded with Excel format")
    

Enter file path(.csv or xlsx:) Input_Deployment.csv


File sucessfully downloaded with CSV format


In [30]:
# Filter 'Forecast' scenario if 'Scenario' column exists
if 'Scenario' in data.columns:
    file_forecast=data[data['Scenario']=='Forecast'][['Month','Revenue','Cost','CCI','CCI %']].copy()
else:
    print("No Forecast data found")

file_forecast['Month'] = pd.to_datetime(file_forecast['Month'])

# Aggregate Revenue and Cost across all clients month-wise
file_forecast = file_forecast.groupby('Month')[['Revenue', 'Cost','CCI']].sum().reset_index()

# Calculate CCI %
file_forecast['CCI %']=((file_forecast['Revenue']-file_forecast['Cost'])/file_forecast['Revenue']*100)

# Sort by Month and set Month as index
file_forecast = file_forecast.sort_values('Month')
file_forecast = file_forecast.set_index('Month')

#Load the mode
filename='model_revenue_forecast.sav'
load_model=pickle.load(open(filename,'rb'))
final_model_revenue=load_model['Revenue']
Ms=load_model['Revenue_Scalar']
#FORECAST & INVERSE TRANSFORM
# ==========================================
steps = len(file_forecast)
forecast_scaled = final_model_revenue.forecast(steps=steps)
revenue_original = Ms.inverse_transform(np.array(forecast_scaled).reshape(-1, 1)).flatten()

#COMPARE FP&A FORECAST VS MODEL PREDICTED REVENUE (CELL 11 LOGIC)
file_forecast.rename(columns={'Revenue': 'FP&A Forecast Revenue'}, inplace=True)

cci_df = pd.DataFrame({
    'Month': file_forecast.index,
    'FP&A Forecast Revenue': file_forecast['FP&A Forecast Revenue'].values,
    'Predicted Revenue': revenue_original
})

cci_df['Difference'] = cci_df['Predicted Revenue'] - cci_df['FP&A Forecast Revenue']
cci_df['Difference%'] = (cci_df['Difference'] / cci_df['FP&A Forecast Revenue']) * 100

print("\n--- FP&A Forecast vs. Model Predicted Revenue ---")
print(cci_df.round(2).to_string(index=False))

# Flag months exceeding +/- 5% variance
attention_months = cci_df[cci_df['Difference%'].abs() > 5]

if not attention_months.empty:
    print("\n⚠️ ATTENTION: Focus on the following months:")
    for month in attention_months['Month']:
        print(f" - {month.strftime('%b-%Y')}: Variance requires attention")
else:
    print("\n✅ All months are within the +/- 5% variance threshold.")

# ==========================================
# 5. SAVE FINAL OUTPUT
# ==========================================
cci_df.round(2).to_csv('revenue_variance_report.csv', index=False)
print("\nSaved to 'revenue_variance_report.csv'")


--- FP&A Forecast vs. Model Predicted Revenue ---
     Month  FP&A Forecast Revenue  Predicted Revenue  Difference  Difference%
2026-01-01            -4058830.58         4063362.25  8122192.82      -200.11
2026-02-01            -4143266.81         4029010.53  8172277.34      -197.24
2026-03-01             4315325.48         4006195.12  -309130.35        -7.16
2026-04-01             4403299.62         3991041.79  -412257.83        -9.36
2026-05-01             4492636.27         3980977.38  -511658.88       -11.39
2026-06-01             1714858.07         3974292.90  2259434.83       131.76
2026-07-01             1779661.91         3969853.26  2190191.35       123.07
2026-08-01             1810518.32         3966904.58  2156386.25       119.10
2026-09-01             4680446.65         3964946.15  -715500.50       -15.29
2026-10-01             4867715.89         3963645.42  -904070.47       -18.57
2026-11-01             5011519.35         3962781.51 -1048737.84       -20.93
2026-12-01   